# E26 — o que se desaprende

A média de tudo é a memória mais antiga que existe, e ela nunca esquece: o peso de cada dia é
1/idade, de modo que **quanto mais o sistema viu, mais lento ele aprende**. A proposição da
meia-vida do capítulo 14 prevê esse caso sem que ninguém o tenha escolhido --- e este caderno
confere a previsão contra as **linhas já publicadas** do próprio capítulo.

In [1]:
# <- brinque com: IDADES, TOLERANCIA, DIAS_DA_SERIE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import esquecimento, graficos

RAIZ = Path.cwd()
IDADES = (1, 2, 5, 10, 21, 50, 250, 1000)
TOLERANCIA = esquecimento.TOLERANCIA_DEGRAU
DIAS_DA_SERIE = 14000

print("frevolab %s | tolerancia %.2f do degrau (a mesma do capitulo 14, na unidade dele)"
      % (frevolab.VERSAO, TOLERANCIA))

frevolab 0.1.0 | tolerancia 0.30 do degrau (a mesma do capitulo 14, na unidade dele)


In [2]:
# A conta da proposicao contra as linhas que o capitulo ja publicou.
publicadas = {"vinte_e_um": (21, 24.69), "cinquenta": (50, 59.59),
              "duzentos_e_cinquenta": (250, 300.4), "mil": (1000, 1203.0)}
linhas = []
for nome, (idade, publicado) in publicadas.items():
    calculado = esquecimento.dias_da_idade(idade, TOLERANCIA)
    linhas.append({"memoria": idade, "publicado": publicado, "calculado": calculado,
                   "diferenca_pp": 100.0 * abs(calculado - publicado) / publicado})
tabela = pd.DataFrame(linhas).set_index("memoria")
print(tabela.round(3).to_string())
print()
print("a formula da idade reproduce as quatro linhas publicadas com erro maximo de %.3f%%"
      % tabela["diferenca_pp"].max())

         publicado  calculado  diferenca_pp
memoria                                    
21           24.69     24.677         0.054
50           59.59     59.595         0.008
250         300.40    300.391         0.003
1000       1203.00   1203.371         0.031

a formula da idade reproduce as quatro linhas publicadas com erro maximo de 0.054%


In [3]:
# O que a idade cobra: os dias para aprender, contra a idade.
idades = np.unique(np.round(np.logspace(0, 4, 60)).astype(int))
dias = np.array([esquecimento.dias_da_idade(int(a), TOLERANCIA) for a in idades])
quadro = pd.DataFrame({"idade": idades, "dias": dias, "razao": dias / idades})
print(quadro.iloc[[0, 10, 20, 30, 40, 50, -1]].round(2).to_string(index=False))
print()
print("aos %d dias de serie a memoria da idade pede %.0f dias para aprender: %.2f vezes a serie"
      % (DIAS_DA_SERIE, esquecimento.dias_da_idade(DIAS_DA_SERIE, TOLERANCIA),
         esquecimento.dias_da_idade(DIAS_DA_SERIE, TOLERANCIA) / DIAS_DA_SERIE))

 idade     dias  razao
     1     0.00   0.00
    12    13.84   1.15
    58    69.23   1.19
   276   331.69   1.20
  1314  1581.42   1.20
  6261  7537.47   1.20
 10000 12039.13   1.20

aos 14000 dias de serie a memoria da idade pede 16855 dias para aprender: 1.20 vezes a serie


In [4]:
# Figura 1: os dias para aprender contra a idade da memoria.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
eixo.loglog(idades, dias, lw=1.9, color="#1f4e79", label="os dias que a memória da idade leva")
eixo.loglog(idades, idades, ls=":", lw=1.3, color="#555555", label="a própria idade")
for idade, publicado in [(21, 24.69), (50, 59.59), (250, 300.4), (1000, 1203.0)]:
    eixo.scatter([idade], [publicado], s=55, marker="o", color="#b03a2e", zorder=5)
eixo.scatter([], [], s=55, marker="o", color="#b03a2e", label="as linhas do capítulo 14")
eixo.set_xlabel("idade da memória (dias de média acumulada)")
eixo.set_ylabel("dias para o erro cruzar a tolerância")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25, which="both", ls=":")
fig.tight_layout()
graficos.salvar(fig, "E26_aprender", 1)
plt.close(fig)
print("figura gravada")

figura gravada


## Leitura visual das figuras

Feita nesta sessão abrindo o .png com a ponte de visão (AGENTS.md §9), depois de o caderno rodar.

O que o desenho mostra: os dois eixos são logarítmicos, e a curva dos dias sobe mais depressa do
que a reta da própria idade --- de modo que a distância entre as duas cresce para a direita. Os
quatro pontos vermelhos caem em cima da curva, sem folga visível, que é o que a conta prometia.

In [5]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "aprender_idades": int(len(IDADES)),
    "aprender_tolerancia": float(TOLERANCIA),
    "aprender_dias_serie": int(DIAS_DA_SERIE),
    "aprender_dias_serie_conta": float(esquecimento.dias_da_idade(DIAS_DA_SERIE, TOLERANCIA)),
    "aprender_razao_serie": float(esquecimento.dias_da_idade(DIAS_DA_SERIE, TOLERANCIA) / DIAS_DA_SERIE),
    "aprender_erro_maximo_pct": float(tabela["diferenca_pp"].max()),
    "aprender_razao_mil": float(esquecimento.dias_da_idade(1000, TOLERANCIA) / 1000.0),
    "aprender_razao_vinte_e_um": float(esquecimento.dias_da_idade(21, TOLERANCIA) / 21.0),
}
NOMES = {1: "um", 2: "dois", 5: "cinco", 10: "dez", 21: "vinte_e_um", 50: "cinquenta",
         250: "duzentos_e_cinquenta", 1000: "mil"}
for idade in IDADES:
    resultado["aprender_dias_%s" % NOMES[idade]] = float(esquecimento.dias_da_idade(idade, TOLERANCIA))
caminho = Path("lab/resultados/E26_aprender.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E26_aprender.json gravado | 16 grandezas
